In [1]:
import pandas as pd
from sklearn.datasets import fetch_california_housing

# Load the dataset
housing = fetch_california_housing(as_frame=True)

# Create a DataFrame
df = housing.frame

# Inspect the data
print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

print("\nColumn information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

Dataset shape: (20640, 9)

First 5 rows:
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  MedHouseVal  
0    -122.23        4.526  
1    -122.22        3.585  
2    -122.24        3.521  
3    -122.25        3.413  
4    -122.25        3.422  

Column information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  flo

In [2]:
from sklearn.model_selection import train_test_split

# Features
X = df.drop(columns=["MedHouseVal"])

# Target
y = df["MedHouseVal"]

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

X shape: (20640, 8)
y shape: (20640,)
Training features: (16512, 8)
Testing features: (4128, 8)
Training target: (16512,)
Testing target: (4128,)


In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# 1. Create the model
model = LinearRegression()

# 2. Train the model
model.fit(X_train, y_train)

# 3. Predict on unseen test data
y_pred = model.predict(X_test)

# 4. Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("MAE:", mae)
print("RMSE:", rmse)

# Compare actual and predicted values
comparison = pd.DataFrame({
    "Actual": y_test.iloc[:10].values,
    "Predicted": y_pred[:10]
})

print(comparison)

MAE: 0.5332001304956553
RMSE: 0.7455813830127764
    Actual  Predicted
0  0.47700   0.719123
1  0.45800   1.764017
2  5.00001   2.709659
3  2.18600   2.838926
4  2.78000   2.604657
5  1.58700   2.011754
6  1.98200   2.645500
7  1.57500   2.168755
8  3.40000   2.740746
9  4.46600   3.915615


In [4]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import pandas as pd

# Baseline model
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)

# Predictions
baseline_pred = baseline.predict(X_test)

# Compare models
results = pd.DataFrame({
    "Model": ["Dummy Baseline", "Linear Regression"],
    "MAE": [
        mean_absolute_error(y_test, baseline_pred),
        mean_absolute_error(y_test, y_pred)
    ],
    "R2": [
        r2_score(y_test, baseline_pred),
        r2_score(y_test, y_pred)
    ]
})

print(results)

               Model       MAE        R2
0     Dummy Baseline  0.906069 -0.000219
1  Linear Regression  0.533200  0.575788


In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# Create the model
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Train
rf_model.fit(X_train, y_train)

# Predict
rf_pred = rf_model.predict(X_test)

# Evaluate
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest Results")
print("MAE:", rf_mae)
print("RMSE:", rf_rmse)
print("R2:", rf_r2)

# Compare all models
comparison = pd.DataFrame({
    "Model": ["Dummy Baseline", "Linear Regression", "Random Forest"],
    "MAE": [
        mean_absolute_error(y_test, baseline_pred),
        mean_absolute_error(y_test, y_pred),
        rf_mae
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_test, baseline_pred)),
        np.sqrt(mean_squared_error(y_test, y_pred)),
        rf_rmse
    ],
    "R2": [
        r2_score(y_test, baseline_pred),
        r2_score(y_test, y_pred),
        rf_r2
    ]
})

print("\nModel Comparison:")
print(comparison.round(4))

Random Forest Results
MAE: 0.32754256845930246
RMSE: 0.5053399773665033
R2: 0.8051230593157366

Model Comparison:
               Model     MAE    RMSE      R2
0     Dummy Baseline  0.9061  1.1449 -0.0002
1  Linear Regression  0.5332  0.7456  0.5758
2      Random Forest  0.3275  0.5053  0.8051


In [6]:
import pandas as pd

feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance.round(4))

      Feature  Importance
0      MedInc      0.5249
5    AveOccup      0.1384
6    Latitude      0.0889
7   Longitude      0.0886
1    HouseAge      0.0546
2    AveRooms      0.0443
4  Population      0.0306
3   AveBedrms      0.0296


In [7]:
errors = pd.DataFrame({
    "Actual": y_test,
    "Predicted": rf_pred
})

errors["Absolute_Error"] = (
    errors["Actual"] - errors["Predicted"]
).abs()

print(
    errors.sort_values(
        by="Absolute_Error",
        ascending=False
    ).head(10).round(3)
)

       Actual  Predicted  Absolute_Error
5887    0.175      3.221           3.046
12069   5.000      1.957           3.043
12389   5.000      2.026           2.974
20349   1.250      4.093           2.843
6688    5.000      2.194           2.806
19542   4.500      1.699           2.801
12220   5.000      2.232           2.768
17306   5.000      2.244           2.756
4548    0.675      3.400           2.725
459     5.000      2.426           2.574


In [8]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor
import numpy as np

rf_cv = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

cv_scores = cross_val_score(
    rf_cv,
    X_train,
    y_train,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

mae_scores = -cv_scores

print("MAE in each fold:", mae_scores.round(4))
print("Mean CV MAE:", round(mae_scores.mean(), 4))
print("Standard Deviation:", round(mae_scores.std(), 4))

MAE in each fold: [0.3405 0.3328 0.3273 0.3312 0.3398]
Mean CV MAE: 0.3343
Standard Deviation: 0.0051


In [9]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "n_estimators": [100, 200],
    "max_depth": [None, 15, 25],
    "min_samples_leaf": [1, 2, 4],
    "max_features": [0.7, 1.0]
}

search = RandomizedSearchCV(
    estimator=RandomForestRegressor(
        random_state=42,
        n_jobs=1
    ),
    param_distributions=param_dist,
    n_iter=10,
    scoring="neg_mean_absolute_error",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search.fit(X_train, y_train)

print("Best Parameters:")
print(search.best_params_)

print("Best CV MAE:")
print(round(-search.best_score_, 4))

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best Parameters:
{'n_estimators': 100, 'min_samples_leaf': 2, 'max_features': 0.7, 'max_depth': 25}
Best CV MAE:
0.3325


In [10]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

best_model = search.best_estimator_

final_pred = best_model.predict(X_test)

final_mae = mean_absolute_error(y_test, final_pred)
final_rmse = np.sqrt(mean_squared_error(y_test, final_pred))
final_r2 = r2_score(y_test, final_pred)

print("Final Test Results")
print("MAE:", round(final_mae, 4))
print("RMSE:", round(final_rmse, 4))
print("R2:", round(final_r2, 4))

Final Test Results
MAE: 0.3267
RMSE: 0.5041
R2: 0.8061
